# 62. Five more verified public members

**One variable against ledger row 145** (`stack_prune35_public`, CV 0.969281): the member set.
Five more candidates, all admitted by the same gate.

## The gate rejected the best-looking vector on the board, which is the point

`writeup/verify_public_oof.py` now screens twelve candidates. Ten pass, two fail, and the two
failures matter more than the ten passes.

| candidate | AUC | author's folds vs ours | verdict |
|---|---|---|---|
| `lookup_srcA` | **0.968691** | **2.17e-03** | **REJECT** |
| `spline_srcD` | 0.966520 | 1.77e-03 | REJECT |
| `lgb_srcB` | 0.968616 | 3.24e-06 | admit |
| `realmlp_srcI` | 0.968138 | 3.60e-06 | admit |
| `hgb_srcH` | 0.968026 | 4.54e-07 | admit |
| `xgb_srcC` | 0.964812 | 3.33e-16 | admit |
| `resnet_kava` | 0.956873 | 4.55e-07 | admit |

**`lookup_srcA` has the highest out-of-fold AUC of any candidate examined, and it is rejected.**
Its per-fold AUCs computed on our partition read 0.96810 where its author printed 0.96593. A
vector that scores *higher* on our folds than on its own is the signature of a different split,
and it is the exact failure this gate exists to catch: the vector is out-of-fold per row, so
nothing about it looks wrong, while the model behind its value on our training rows saw rows
inside our validation fold.

Admitting it would have raised CV and it would not have raised the leaderboard by the same
amount. The offset check on row 145 came in at +0.001259 against a band of +0.001256 to
+0.001322, and that is the number a clean pool produces.

A second thing the gate caught: srcB publishes an OOF *dataset* as well as a kernel, and the two
are different vectors at Spearman 0.9929. The kernel output verifies; the dataset copy has no
printed fold AUCs and is therefore rejected. Admitting it as "the same model" would have been
wrong on the evidence.

## What is deliberately absent

No blends, no hill-climbing ensembles, no other people's level-2 stacks, though several are
public and several score higher than anything here. Their weights may have been chosen against
public leaderboard feedback, which is the mechanism srcP's analysis identifies as the
thing that manufactures a public fifth decimal and loses it on the private split. Stacking a
stack inherits that invisibly. **Base models only.**

## The prediction

**+0.00015 to +0.00035.** Row 145's five bought +0.000341, and these five are individually
stronger on average, but the pool is now sixty members deep and the first five already took the
weight that was going spare. `lgb_srcB` at 0.968616 is the strongest single vector in the whole
stack including our own.

## The case against, written first

**Saturation, and it is the same argument that has been right eleven times.** The first five
public members displaced 94 percent of their own weight from existing members rather than adding
new information. Five more drawn from the same public pool are more of the same direction, and
`realmlp_srcI` in particular is a second implementation of a class the stack now holds three
times.

**`xgb_srcC` at 0.964812 and `resnet_kava` at 0.956873 are well behind the field.** Row 142
measured what a member 0.0155 behind contributes, and the answer was exactly zero. `resnet_kava`
is 0.0124 behind `lgb_srcB`. If the row 142 bound holds, it contributes nothing and is in the set
only because the gate cleared it.

## Where this leaves the target

Top 10 percent needs public LB 0.97103, which on the realised +0.001259 offset is **CV 0.969771**,
or +0.00049 above row 145. The prediction above does not reach it. If this lands mid-range the
gap after it is roughly +0.0002, and the honest read is that the remaining distance has to come
from more members rather than from anything cleverer.


In [1]:
# 37_stack_views.ipynb
# Membership gate for the five vectors produced by notebooks 35 and 36.
# Runs locally: every member vector already lives in artifacts/oof.
import hashlib
import json
import pathlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(b for b in [Path.cwd(), *Path.cwd().parents]
            if (b / "data" / "raw" / "train.csv").exists())
O = ROOT / "artifacts" / "oof"
S = ROOT / "submissions"

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy(np.int8)

# The fold vector is rebuilt rather than loaded, and then checked. A silently different
# fold vector is the one error here that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
FOLD_SHA = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
assert FOLD_SHA == "ec282b0968059676", FOLD_SHA
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")
print(f"fold sha {FOLD_SHA}  VERIFIED")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]
fold sha ec282b0968059676  VERIFIED


In [2]:
# Row 94's forty-one minus the duplicate, in row 94's order, then the five candidates.
BASE = [
    ("te42", "te_bag42"), ("te2024", "te_seed2024"), ("te7", "te_seed7"),
    ("te2025", "te_seed2025"), ("te13", "te_seed13"),
    ("anchor", "lgbm_default_anchor_seed42"), ("trees300", "lgbm_trees300_seed42"),
    ("trees1000", "lgbm_trees1000_seed42"), ("trees2000", "lgbm_trees2000_seed42"),
    ("lr010", "lgbm_lr01_n1000_seed42"), ("lr005", "lgbm_lr005_n2000_seed42"),
    ("lr003", "lgbm_lr003_n3333_seed42"),
    ("bag42", "lgbm_bag08_lr005_n2000_seed42"),
    ("bag2024", "lgbm_bag08_lr005_n2000_seed2024"),
    ("bag7", "lgbm_bag08_lr005_n2000_seed7"),
    ("bag2025", "lgbm_bag08_lr005_n2000_seed2025"),
    ("bag13", "lgbm_bag08_lr005_n2000_seed13"),
    ("neural", "neural"), ("cat42", "catboost_te"), ("cat2024", "catboost_te_seed2024"),
    ("cat7", "catboost_te_seed7"), ("cat2025", "catboost_te_seed2025"),
    ("cat13", "catboost_te_seed13"), ("neural_te", "neural_te"),
    ("xgb_te", "xgb_te"), ("xgb2024", "xgb_te_seed2024"), ("xgb7", "xgb_te_seed7"),
    ("xgb2025", "xgb_te_seed2025"), ("xgb13", "xgb_te_seed13"),
    ("pair_top9", "xgb_pair_top9"),
]
BASE += [("cat_nat_c1", "cat_native_c1"), ("cat_nat_c2", "cat_native_c2"),
         ("xgb_raw", "xgb_raw"), ("cat_raw", "cat_raw"),
         ("xgb_te_fe", "xgb_te_fe"), ("cat_te_n4000", "cat_te_n4000"),
         ("xgb_raw_fe", "xgb_raw_fe"), ("cat_raw_n10k", "cat_raw_n10000"),
         ("lgb_raw_fe", "lgb_raw_fe"), ("cat_raw_fe", "cat_raw_fe")]
# lgb_raw is deliberately absent: it is bag42 re-run in another kernel, Pearson 0.999970 on
# logits. See the header and the 2026-08-22 entry in NOTES.md. Dropping it costs -0.000001.
DROPPED = [("lgb_raw", "duplicate of bag42, Pearson 0.999970")]

BASE += [("cat_te_fe", "cat_te_fe"), ("hgb_te_fe", "hgb_te_fe"),
         ("lgb_te_fe", "lgb_te_fe"), ("rf_te_fe", "rf_te_fe"),
         ("neural_lookup", "neural_lookup"), ("et_te_fe", "et_te_fe"),
         ("logit_te_fe", "logit_te_fe")]
BASE += [("xgb_tuned", "xgb_tuned")]
BASE += [("neural_fe", "neural_fe"), ("neural_wide", "neural_wide"),
         ("neural_res", "neural_res")]
BASE += [("realmlp", "realmlp")]
BASE += [("realmlp10", "realmlp10")]
BASE += [("realmlp_raw_fe", "realmlp_raw_fe")]
BASE += [("tabm", "tabm")]
# THE ONE VARIABLE. Row 144 held these fifty-five, all built by this repo. The five
# candidates below were not. Each is admitted only by writeup/verify_public_oof.py,
# which proves the fold partition matches ours by reproducing the author's own printed
# per-fold AUCs from our fold vector. See the header.
# Row 145's five, now part of the base.
PRIOR_PUBLIC = ["cb_srcE", "lgb_srcE", "xgb_srcA", "realmlp_srcA", "tabm_srcA"]
# THE ONE VARIABLE. Five more, admitted by the same gate.
NEW_PUBLIC = ["lgb_srcB", "realmlp_srcI", "hgb_srcH", "xgb_srcC", "resnet_kava"]
PUBLIC = PRIOR_PUBLIC + NEW_PUBLIC
CAND = [(n, n) for n in NEW_PUBLIC]


def load(stem, kind):
    # OOF/test vector. Two naming conventions exist in artifacts/oof. The bare
    # "{stem}.npy" form is the OOF side only: the early LightGBM members never had a
    # test .npy written and their test side lives in submissions/. Falling back to the
    # bare name for kind="test" silently returns the OOF vector, which is caught by the
    # length assert below only because train and test differ in length.
    cands = [O / f"{stem}_{kind}.npy"]
    if kind == "oof":
        cands.append(O / f"{stem}.npy")
    for c in cands:
        if c.exists():
            return np.load(c)
    if kind == "test" and (S / f"{stem}.csv").exists():
        df = pd.read_csv(S / f"{stem}.csv")
        # A csv written in a different row order blends perfectly cleanly and is
        # undetectable in the score. Checked rather than assumed.
        assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {stem}"
        return df["addicted_label"].to_numpy()
    raise FileNotFoundError(f"{stem} {kind}")


# Row 145 already holds PRIOR_PUBLIC, so they belong in the index alongside our own.
MEM = BASE + [(n, n) for n in PRIOR_PUBLIC] + CAND
names = [n for n, _ in MEM]
Poof = {n: load(s, "oof") for n, s in BASE}
Ptest = {n: load(s, "test") for n, s in BASE}

# The public five, read from artifacts/public_oof/ with their id order asserted rather
# than assumed. A csv in a different row order blends perfectly cleanly and is invisible
# in the score, which is the failure row 59's loader already guards against.
PUB = ROOT / "artifacts" / "public_oof"
VER = {r["name"]: r for r in json.loads((PUB / "verification.json").read_text(encoding="utf-8"))}


def read_vec(path, n_expected, order_ref):
    """Mirror of writeup/verify_public_oof.load_vector, so a member is loaded here
    exactly as it was loaded when it was verified. Two shapes exist in the wild: a
    csv with or without an id column, and a bare .npy."""
    path = pathlib.Path(path)
    if path.suffix == ".npy":
        v = np.load(path)
        assert len(v) == n_expected, f"{path.name} has {len(v)} rows"
        return v.astype(float)
    df = pd.read_csv(path)
    assert len(df) == n_expected, f"{path.name} has {len(df)} rows"
    idc = [c for c in df.columns if c.lower() == "id"]
    if idc:
        # A csv in a different row order blends perfectly cleanly and is invisible in
        # the score, so this is asserted rather than hoped for.
        assert (df[idc[0]].to_numpy() == order_ref).all(), f"id order {path.name}"
    pref = [c for c in df.columns if any(k in c.lower() for k in ("oof", "pred", "prob"))]
    col = pref or [c for c in df.columns
                   if c.lower() not in ("id", "addicted_label", "target", "fold")]
    col = col or [c for c in df.columns if c.lower() != "id"]
    return df[col[0]].to_numpy(float)


for n in PUBLIC:
    r = VER.get(n)
    assert r and r["verdict"].startswith("ADMISSIBLE"), \
        f"{n} is not admissible: {r['verdict'] if r else 'absent'}. Re-run the gate."
    d = PUB / r["folder"]
    Poof[n] = read_vec(d / r["oof_file"], len(train), train["id"].to_numpy())
    Ptest[n] = read_vec(d / r["test_file"], len(test), test["id"].to_numpy())
print(f"{len(PUBLIC)} public members loaded, every one verified on fold protocol")
print(f"  carried from row 145: {PRIOR_PUBLIC}")
print(f"  new this run        : {NEW_PUBLIC}")
_rej = [k for k, v in VER.items() if not v["verdict"].startswith("ADMISSIBLE")]
print(f"  rejected by the gate: {_rej}")

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    assert np.isfinite(Poof[n]).all() and np.isfinite(Ptest[n]).all(), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

# Exact-duplicate quarantine. A duplicated array silently DOUBLES that model's weight.
# Row 59 found xgb_pair_base bit-identical to xgb_te this way and excluded it.
seen = {}
for n in names:
    h = hashlib.md5(np.ascontiguousarray(Poof[n]).tobytes()).hexdigest()
    assert h not in seen, f"{n} is bit-identical to {seen[h]}"
    seen[h] = n
print(f"{len(names)} vectors loaded, no exact duplicates")

# THE CORRELATION SCREEN. Hash equality cannot see one configuration run in two kernels:
# the arrays differ by thread-level numerical noise. bag42 and lgb_raw sat in row 94 at
# Pearson 0.999970 and no check fired. Three lines, and it would have caught them.
Lz = np.column_stack([np.clip(np.log(np.clip(Poof[n], 1e-9, 1 - 1e-9)
                                     / (1 - np.clip(Poof[n], 1e-9, 1 - 1e-9))), -30, 30)
                      for n in names])
Cm = np.corrcoef(((Lz - Lz.mean(0)) / Lz.std(0)).T)
np.fill_diagonal(Cm, 0.0)
near = [(names[i], names[j], Cm[i, j])
        for i in range(len(names)) for j in range(i + 1, len(names))
        if abs(Cm[i, j]) > 0.9999]
print(f"pairs above 0.9999: {len(near)}")
for a, b, r in near:
    print(f"  NEAR-DUPLICATE {a} and {b} at {r:+.6f}")
assert not near, "a near-duplicate pair is present, justify it or drop one"
print(f"removed from row 94's set: {[d[0] for d in DROPPED]}")
hi = sorted(((abs(Cm[i, j]), names[i], names[j])
             for i in range(len(names)) for j in range(i + 1, len(names))),
            reverse=True)[:3]
print("most collinear surviving pairs: "
      + ", ".join(f"{a}/{b} {r:.5f}" for r, a, b in hi))

10 public members loaded, every one verified on fold protocol
  carried from row 145: ['cb_srcE', 'lgb_srcE', 'xgb_srcA', 'realmlp_srcA', 'tabm_srcA']
  new this run        : ['lgb_srcB', 'realmlp_srcI', 'hgb_srcH', 'xgb_srcC', 'resnet_kava']
  rejected by the gate: ['lookup_srcA', 'spline_srcD']


65 vectors loaded, no exact duplicates


pairs above 0.9999: 0
removed from row 94's set: ['lgb_raw']
most collinear surviving pairs: realmlp/realmlp10 0.99948, cat42/cat7 0.99907, cat2024/cat2025 0.99907


In [3]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
IDX = {n: i for i, n in enumerate(names)}
BASE60 = [IDX[n] for n, _ in BASE] + [IDX[n] for n in PRIOR_PUBLIC]

print("candidate solo CV, and disagreement with the members it most resembles:")
print(f"  {'candidate':12} {'solo CV':>10} {'rho vs xgb_te':>15} {'rho vs cat42':>14}")
for n, _ in CAND:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    r1 = pd.Series(Poof[n]).corr(pd.Series(Poof["xgb_te"]), method="spearman")
    r2 = pd.Series(Poof[n]).corr(pd.Series(Poof["cat42"]), method="spearman")
    print(f"  {n:12} {cv:10.6f} {r1:15.6f} {r2:14.6f}")

# For scale: how decorrelated are two members that everyone agrees are near-copies?
r_seed = pd.Series(Poof["xgb_te"]).corr(pd.Series(Poof["xgb2024"]), method="spearman")
print(f"\n  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): {r_seed:.6f}")
print("  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.")
print("  This table is context, not a prediction.")

candidate solo CV, and disagreement with the members it most resembles:
  candidate       solo CV   rho vs xgb_te   rho vs cat42


  lgb_srcB       0.968617        0.994351       0.991284


  realmlp_srcI   0.968176        0.976500       0.975814


  hgb_srcH       0.968027        0.992750       0.988395


  xgb_srcC     0.964814        0.983511       0.977780


  resnet_kava    0.956995        0.965590       0.958824



  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): 0.997344
  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.
  This table is context, not a prediction.


In [4]:
def run(cols):
    # Fold-wise logistic combiner. No weight is ever fitted on a row it is scored on.
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    nit = []
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)], y[tr])
        # A non-converged lbfgs fit reads HIGHER than the truth, so convergence is
        # asserted rather than hoped for. Added 2026-08-21 after the public review
        # flagged it; measured at 38 to 40 iterations, so it has never been close.
        nit.append(int(np.max(clf.n_iter_)))
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    assert max(nit) < 2000, f"combiner did not converge, {nit}"
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf, max(nit)


ARMS = {"60_row145": BASE60}
for n, _ in CAND:
    ARMS[f"61_{n}"] = BASE60 + [IDX[n]]
ARMS["61_all5"] = BASE60 + [IDX[n] for n, _ in CAND]

res = {a: run(cols) for a, cols in ARMS.items()}

# The prune, recomputed inside a committed notebook rather than trusted from the audit's
# scratch script. Keeps the top k members of the WINNING arm by absolute coefficient.
_best_cols = ARMS["61_all5"]
_coef = res["61_all5"][2].mean(axis=0)
_order = np.argsort(-np.abs(_coef))
for _k in (16, 25, 35):
    if _k < len(_best_cols):
        ARMS[f"prune{_k}"] = sorted(_best_cols[i] for i in _order[:_k])
        res[f"prune{_k}"] = run(ARMS[f"prune{_k}"])
per = {a: r[0] for a, r in res.items()}

# Row 54 compared its base arm against 0.968932, which is row 137's PRUNED value rather
# than the 53-member figure. That labelling slip is recorded in row 139's notes and is
# corrected here: the base arm below IS row 139's fifty-three, and 0.968932 is its CV.
ROW145_CV, ROW140_CV = 0.969281, 0.969281
repro = per["60_row145"].mean() - ROW145_CV
print(f"reproduction of row 145: {per['60_row145'].mean():.6f} vs {ROW145_CV:.6f}"
      f"  delta {repro:+.2e}   {'REPRODUCED' if abs(repro) < 1e-4 else 'FAILED'}")
assert abs(repro) < 1e-4, "base arm does not reproduce row 145, do not log this run"
print(f"combiner max n_iter across all arms: {max(r[3] for r in res.values())} of 2000\n")

hdr = " ".join(f"{'fold ' + str(i):>9}" for i in range(5))
print(f"{'arm':14} {hdr} {'mean':>10} {'sd':>9}")
for a in ARMS:
    print(f"{a:14} " + " ".join(f"{v:9.6f}" for v in per[a])
          + f" {per[a].mean():10.6f} {per[a].std():9.6f}")

reproduction of row 145: 0.969275 vs 0.969281  delta -6.26e-06   REPRODUCED
combiner max n_iter across all arms: 70 of 2000

arm               fold 0    fold 1    fold 2    fold 3    fold 4       mean        sd
60_row145       0.968624  0.969356  0.969442  0.969886  0.969065   0.969275  0.000418
61_lgb_srcB     0.968644  0.969389  0.969460  0.969918  0.969102   0.969303  0.000421
61_realmlp_srcI  0.968722  0.969458  0.969473  0.969987  0.969180   0.969364  0.000414
61_hgb_srcH     0.968630  0.969363  0.969443  0.969886  0.969077   0.969280  0.000416
61_xgb_srcC   0.968629  0.969353  0.969442  0.969893  0.969061   0.969276  0.000419
61_resnet_kava  0.968629  0.969375  0.969450  0.969890  0.969068   0.969282  0.000419
61_all5         0.968745  0.969501  0.969502  0.970017  0.969223   0.969398  0.000415
prune16         0.968731  0.969490  0.969513  0.970016  0.969209   0.969392  0.000421
prune25         0.968741  0.969509  0.969514  0.970004  0.969222   0.969398  0.000414
prune35         

In [5]:
FLOOR_MEAN, FLOOR_FOLDS = 0.00005, 4
# THE LEAK TRIPWIRE. NOTES.md: a feature that jumps CV by an implausible amount is a
# leak until proven otherwise. A verified partition should behave like any other member
# set; anything above +0.002 here means the verification missed something.
LEAK_ALARM = 0.002
base_per = per["60_row145"]

print("Paired against row 145's sixty. The gate is >= +0.00005 mean AND >= 4/5 folds.\n")
print(f"{'arm':14} {'paired mean':>13} {'paired sd':>11} {'folds':>7} {'t(4)':>8}  gate")
gate = {}
for a in ARMS:
    if a == "60_row145":
        continue
    d = per[a] - base_per
    wins = int((d > 0).sum())
    sd = d.std(ddof=1)
    t = d.mean() / (sd / np.sqrt(5)) if sd > 0 else float("inf")
    fired = bool(d.mean() >= FLOOR_MEAN and wins >= FLOOR_FOLDS)
    gate[a] = fired
    print(f"{a:14} {d.mean():+13.6f} {sd:11.6f} {wins:5d}/5 {t:8.2f}"
          f"  {'FIRES' if fired else 'under floor'}")
    assert d.mean() < LEAK_ALARM, (
        f"{a} gains {d.mean():+.6f}, above the {LEAK_ALARM} alarm. Treat as a leak and "
        f"re-run writeup/verify_public_oof.py before believing this.")

Paired against row 145's sixty. The gate is >= +0.00005 mean AND >= 4/5 folds.

arm              paired mean   paired sd   folds     t(4)  gate
61_lgb_srcB        +0.000028    0.000008     5/5     7.44  under floor
61_realmlp_srcI     +0.000089    0.000034     5/5     5.95  FIRES
61_hgb_srcH        +0.000005    0.000005     4/5     2.21  under floor
61_xgb_srcC      +0.000001    0.000004     2/5     0.40  under floor
61_resnet_kava     +0.000008    0.000007     5/5     2.59  under floor
61_all5            +0.000123    0.000038     5/5     7.27  FIRES
prune16            +0.000117    0.000029     5/5     8.98  FIRES
prune25            +0.000123    0.000034     5/5     8.03  FIRES
prune35            +0.000131    0.000036     5/5     8.16  FIRES


In [6]:
# Coefficients of the best arm, which is where the result actually lives. Row 59's
# lesson: a model that is null on its own can still take a large weight, and where that
# weight comes FROM is the thing worth reading.
best = max((a for a in ARMS if a != "60_row145"), key=lambda a: per[a].mean())
print(f"best arm by CV: {best}   {per[best].mean():.6f}\n")

cols_b, cols_0 = ARMS[best], ARMS["60_row145"]
cb = res[best][2].mean(axis=0)
c0 = res["60_row145"][2].mean(axis=0)
base_map = {names[c]: c0[i] for i, c in enumerate(cols_0)}

rows = []
for i, c in enumerate(cols_b):
    n = names[c]
    was = base_map.get(n, float("nan"))
    rows.append({"member": n, "coef": cb[i], "was": was, "shift": cb[i] - was})
tab = pd.DataFrame(rows).sort_values("coef", ascending=False)
print(tab.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

new = set(n for n, _ in CAND) & set(tab.member)
gained = tab[tab.member.isin(new)]["coef"].sum()
lost = -tab[~tab.member.isin(new)]["shift"].sum()
print(f"\nnew members carry {gained:+.4f} in total")
print(f"the existing fifty-three give up {lost:+.4f} of weight between them")
if abs(gained) > 1e-12:
    print(f"substitution covers {100 * lost / gained:.0f} percent of the new weight")

best arm by CV: prune35   0.969406

       member    coef     was   shift
 realmlp_srcI +0.2615     NaN     NaN
   cat_nat_c2 +0.2584 +0.2607 -0.0023
     lgb_srcB +0.2286     NaN     NaN
 realmlp_srcA +0.2200 +0.3737 -0.1538
      lgb_srcE +0.1339 +0.1781 -0.0442
 cat_te_n4000 +0.0910 +0.1046 -0.0136
       cb_srcE +0.0812 +0.1034 -0.0222
    realmlp10 +0.0760 +0.0918 -0.0159
   xgb_srcC +0.0692     NaN     NaN
   xgb_raw_fe +0.0450 +0.0876 -0.0426
    hgb_te_fe +0.0440 +0.0366 +0.0074
   neural_res +0.0413 +0.0442 -0.0029
  resnet_kava +0.0412     NaN     NaN
    cat_te_fe +0.0407 +0.0324 +0.0082
   lgb_raw_fe +0.0350 +0.0377 -0.0027
        lr005 +0.0290 +0.0319 -0.0029
      xgb2024 +0.0286 +0.0329 -0.0043
         xgb7 +0.0223 +0.0226 -0.0003
    trees1000 +0.0213 +0.0188 +0.0024
neural_lookup +0.0212 +0.0323 -0.0111
     rf_te_fe -0.0250 -0.0330 +0.0081
  neural_wide -0.0271 -0.0372 +0.0101
     xgb_srcA -0.0308 +0.0201 -0.0509
    neural_fe -0.0327 -0.0377 +0.0050
        bag42 

In [7]:
# Three decisions, kept separate. Bundling them was the error corrected in row 59.
print("1. GATE")
for a, f in gate.items():
    print(f"     {a:14} {'FIRES' if f else 'under floor'}")

print("\n2. MEMBERSHIP")
print("   A sub-floor addition is still kept and logged as negligible: row 32 kept four")
print("   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership")
print("   follows the sign and the fold count, not the floor.")
keep = [a for a in ARMS if a != "60_row145"
        and (per[a] - base_per).mean() > 0
        and int(((per[a] - base_per) > 0).sum()) >= 4]
print(f"   arms positive and >= 4/5 folds: {keep if keep else 'none'}")
print(f"   carried forward: {best} at {per[best].mean():.6f}")

print("\n3. SUBMISSION")
SUB = S / "stack_public2.csv"
# The floor the gate uses, applied to the submission decision too. Row 142 recorded
# that these had different thresholds and that a two-millionth difference wrote a csv.
if per[best].mean() > ROW140_CV + FLOOR_MEAN:
    p = res[best][1].mean(axis=0)
    sub = pd.DataFrame({"id": test["id"].to_numpy(),
                        "addicted_label": (np.argsort(np.argsort(p)) + 0.5) / len(p)})
    assert len(sub) == len(test) and np.isfinite(sub["addicted_label"]).all()
    sub.to_csv(SUB, index=False)
    print(f"   wrote {SUB.name}, {len(sub):,} rows,"
          f" {sub['addicted_label'].nunique():,} distinct")
    print("   AUC reads order only, so the rank transform changes nothing and keeps the")
    print("   file comparable with the earlier stack submissions.")
else:
    print(f"   no submission: best arm {per[best].mean():.6f} does not beat row 140's "
          f"{ROW140_CV:.6f}")

print(f"\nledger lines:\n  name    stack_{best}\n  cv_mean {per[best].mean():.6f}"
      f"\n  cv_std  {per[best].std():.6f}")

1. GATE
     61_lgb_srcB    under floor
     61_realmlp_srcI FIRES
     61_hgb_srcH    under floor
     61_xgb_srcC  under floor
     61_resnet_kava under floor
     61_all5        FIRES
     prune16        FIRES
     prune25        FIRES
     prune35        FIRES

2. MEMBERSHIP
   A sub-floor addition is still kept and logged as negligible: row 32 kept four
   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership
   follows the sign and the fold count, not the floor.
   arms positive and >= 4/5 folds: ['61_lgb_srcB', '61_realmlp_srcI', '61_hgb_srcH', '61_resnet_kava', '61_all5', 'prune16', 'prune25', 'prune35']
   carried forward: prune35 at 0.969406

3. SUBMISSION


   wrote stack_public2.csv, 296,302 rows, 296,302 distinct
   AUC reads order only, so the rank transform changes nothing and keeps the
   file comparable with the earlier stack submissions.

ledger lines:
  name    stack_prune35
  cv_mean 0.969406
  cv_std  0.000412
